# medguard — Healthcare LLM Guardrails Demo

This notebook walks through **medguard**'s five safety layers with real examples.

No LLM API key needed for most cells — only the full chat demo at the end requires `ANTHROPIC_API_KEY`.

```
pip install medguard
```

In [ ]:
# Install if needed
# !pip install medguard

---
## 1. PHI Detection & Redaction

medguard strips Protected Health Information before it ever reaches the LLM.

In [ ]:
from medguard.guardrails.phi import PHIDetector
from medguard.config import PHIConfig

detector = PHIDetector(PHIConfig(enabled=True, mode="redact", engine="regex"))

text = "Patient John Smith, DOB 1985-03-15, SSN 123-45-6789, MRN #A4921083. "\
       "Contact: john.smith@email.com or (555) 867-5309."

result = detector.detect(text)

print("Original:", result.original)
print()
print("Redacted:", result.processed)
print()
print(f"PHI detected: {result.phi_detected} ({len(result.matches)} entities)")
print()
for m in result.matches:
    print(f"  [{m.entity_type}] '{m.text}' → '{m.redacted_text}' (confidence: {m.confidence:.0%})")

---
## 2. Drug-Drug Interaction Check

Detects dangerous combinations in free-text queries — before sending them to the LLM.

In [ ]:
import asyncio
from medguard.guardrails.drug_safety import DrugSafetyChecker
from medguard.config import DrugSafetyConfig

# Uses the bundled static table — no network required
config = DrugSafetyConfig(enabled=True, use_openfda=False, severity_threshold="moderate")
checker = DrugSafetyChecker(config, rxnorm=None, openfda=None)

dangerous_query = "My patient is on warfarin 5mg daily. Can I also prescribe aspirin 325mg?"
result = asyncio.run(checker.check(dangerous_query))

print(f"Drugs found: {[d.raw_name for d in result.drugs_found]}")
print(f"Interactions: {len(result.interactions)}")
print(f"Highest severity: {result.highest_severity}")
print(f"Blocked: {result.blocked}")
print()
for interaction in result.interactions:
    print(f"  ⚠️  {interaction.drug_a} + {interaction.drug_b}: {interaction.severity.value.upper()}")
    print(f"     {interaction.description}")

In [ ]:
# The deadliest combination: SSRI + MAOI (serotonin syndrome)
result2 = asyncio.run(checker.check("Patient takes sertraline 100mg. Starting phenelzine."))

print(f"Severity: {result2.highest_severity}")
print(f"Blocked: {result2.blocked}")
for i in result2.interactions:
    print(f"\n  {i.drug_a} + {i.drug_b} → {i.severity.value.upper()}")
    print(f"  {i.description}")

---
## 3. Clinical Scope Enforcement

Detects when a user asks the medical AI legal or financial questions it shouldn't answer.

In [ ]:
from medguard.guardrails.scope import ScopeEnforcer
from medguard.config import ScopeConfig

enforcer = ScopeEnforcer(ScopeConfig(enabled=True, action="warn"))

queries = [
    ("What are the side effects of metformin?",           "✅ clinical"),
    ("I have chest pain and shortness of breath.",        "✅ clinical"),
    ("Can I sue my doctor for malpractice?",              "❌ legal"),
    ("Will my insurance cover this surgery?",             "❌ financial"),
    ("What is the max dose of ibuprofen for adults?",     "✅ clinical"),
]

for query, expected in queries:
    result = enforcer.check(query)
    status = "PASS" if result.in_scope else f"BLOCKED — {result.category.value}"
    print(f"{expected}  [{status}]  '{query[:55]}...' " if len(query) > 55 else f"{expected}  [{status}]  '{query}'")

---
## 4. Hallucination Detection

Catches fake drug names, impossible dosages, and overconfident claims in LLM *output*.

In [ ]:
from medguard.guardrails.hallucination import HallucinationDetector, _compute_hallucination_score
from medguard.guardrails.hallucination import _check_confident_claims_sync, _CONFIDENT_RE
from medguard.config import HallucinationConfig

# Test the dosage checker directly (no network needed)
from medguard.guardrails.hallucination import HallucinationDetector
from medguard.config import HallucinationConfig, DrugSafetyConfig
from unittest.mock import AsyncMock, MagicMock

config = HallucinationConfig(
    enabled=True,
    check_drug_names=False,  # skip RxNorm network call for demo
    check_dosages=True,
    check_confident_claims=True,
    confidence_threshold=0.7,
)

# Mock RxNorm and SNOMED
rxnorm = MagicMock()
rxnorm.validate_drug_exists = AsyncMock(return_value=True)
snomed = MagicMock()
snomed.concept_exists = AsyncMock(return_value=True)

detector = HallucinationDetector(config, rxnorm=rxnorm, snomed=snomed)

# LLM output with two problems: impossible dose + overconfident claim
bad_llm_output = (
    "You should take ibuprofen 10000mg daily for your pain. "
    "This will definitely cure your inflammation completely and is absolutely safe."
)

result = asyncio.run(detector.check(bad_llm_output))

print(f"Hallucination score: {result.hallucination_score:.2f} / 1.0")
print(f"Blocked: {result.blocked}")
print(f"Flags: {len(result.flags)}")
print()
for flag in result.flags:
    print(f"  🚩 [{flag.type.value}] '{flag.text}'")
    print(f"     {flag.explanation}")
    print()
print("Annotated output:")
print(result.annotated_text)

---
## 5. Full Pipeline — check() without an LLM

`mg.check()` runs all input guardrails synchronously — useful for validation without LLM cost.

In [ ]:
from medguard import MedGuard

mg = MedGuard()  # loads ~/.medguard/config.json or uses defaults

# Example: patient note with PHI + dangerous drug combo
note = "Patient Jane Doe (SSN 987-65-4321) is taking warfarin and just started ibuprofen 800mg TID."

result = mg.check(note)

print("=== PHI ===")
print(f"Detected: {result.phi_result.phi_detected}")
print(f"Redacted: {result.phi_result.processed}")
print()
print("=== Drug Safety ===")
if result.drug_result:
    print(f"Highest severity: {result.drug_result.highest_severity}")
    for i in result.drug_result.interactions:
        print(f"  {i.drug_a} + {i.drug_b}: {i.severity.value}")
print()
print("=== Pipeline ===")
print(f"Blocked: {result.blocked}")
print(f"Warnings: {result.warnings}")

---
## 6. Guardrailed LLM Chat (requires ANTHROPIC_API_KEY)

The full pipeline: PHI redaction → scope check → drug safety → LLM → hallucination detection.

In [ ]:
import os

# Set your API key
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("Set ANTHROPIC_API_KEY to run this cell.")
else:
    mg = MedGuard()
    response = asyncio.run(mg.achat("What are the side effects of metformin for type 2 diabetes?"))

    print("Response:", response.processed_output)
    print()
    print("Guardrails triggered:", response.warnings or "none")
    print("Blocked:", response.blocked)

---
## 7. Dangerous query — watch the pipeline block it

In [ ]:
if not os.environ.get("ANTHROPIC_API_KEY"):
    print("Set ANTHROPIC_API_KEY to run this cell.")
else:
    # This should trigger drug safety warning before reaching the LLM
    response = asyncio.run(mg.achat(
        "I'm on warfarin 5mg. My doctor said I can take ibuprofen 800mg three times a day for my back pain."
    ))

    print("Blocked:", response.blocked)
    print("Block reason:", response.block_reason)
    print("Warnings:", response.warnings)

---
## Summary

| Layer | What it caught |
|-------|---------------|
| PHI Detection | SSN, DOB, name, email, phone, MRN |
| Drug Safety | warfarin+aspirin (HIGH), warfarin+ibuprofen (HIGH), sertraline+phenelzine (CONTRAINDICATED) |
| Scope Enforcement | Legal and insurance queries |
| Hallucination Detection | Impossible dosage (ibuprofen 10000mg), overconfident claims |

Each layer runs in an isolated try/except — an API timeout in drug safety never blocks the full request.

**Links:**
- GitHub: https://github.com/sarvanithin/Medguard
- Docs: `pip install medguard` then `medguard serve`
- API: `POST /v1/chat`, `POST /v1/check/phi`, `POST /v1/check/drug-interactions`